In [2]:
import cv2
import pandas as pd
import numpy as np

# ---------- PARAMÈTRES ----------
video_path = "/Volumes/My Passport/PRIMARY/323v_2023-10-29 12_52_08_323v_3_54_4_20_canny.mp4"
csv_path = "/Volumes/My Passport/lanes/323v_2023-10-29 12_52_08_323v_3_54_4_20_segformer_pixels.csv"
output_path = "output_overlay.mp4"

# Couleur du masque (BGR) : rouge
mask_color = (0, 0, 255)
alpha = 0.4  # transparence

# ---------- LECTURE CSV ----------
df = pd.read_csv(csv_path)

# Convertir le CSV en dictionnaire : frame_id -> liste de pixels [(x,y), ...]
frame_pixels = {}

for _, row in df.iterrows():
    frame_id = int(row["frame"])
    pixel_str = str(row["pixels"])

    coords = []
    coords = []

    for p in pixel_str.split(";"):
        p = p.strip()

        if "_" not in p:
            continue

        try:
            x_str, y_str = p.split("_")
            x, y = int(x_str), int(y_str)
            coords.append((x, y))
        except:
            continue
    frame_pixels[frame_id] = coords

# ---------- LECTURE VIDÉO ----------
cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    raise ValueError("Impossible d'ouvrir la vidéo.")

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)
frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

frame_idx = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break

    overlay = frame.copy()

    if frame_idx in frame_pixels:
        # masque vide
        mask = np.zeros((height, width), dtype=np.uint8)

        for x, y in frame_pixels[frame_idx]:
            if 0 <= x < width and 0 <= y < height:
                mask[y, x] = 255   # attention: image = [y, x]

        # colorer les pixels détectés
        overlay[mask == 255] = mask_color

        # fusion transparente
        frame = cv2.addWeighted(overlay, alpha, frame, 1 - alpha, 0)

    out.write(frame)
    frame_idx += 1

cap.release()
out.release()
print(f"Vidéo sauvegardée : {output_path}")

Vidéo sauvegardée : output_overlay.mp4


In [5]:
import xml.etree.ElementTree as ET
from collections import defaultdict
import re

# Chemins
kml_paths = {
    "vague1": "/Volumes/My Passport/all_trips_vague1.kml",
    "vague2": "/Volumes/My Passport/all_trips_vague2.kml",
    "vague3": "/Volumes/My Passport/all_trips_vague3.kml",
}

ns = 'http://www.opengis.net/kml/2.2'

all_seconds = defaultdict(int)  # clé : (vague, trip_id)

for vague, kml_path in kml_paths.items():
    print(f"Traitement {vague}...")
    tree = ET.parse(kml_path)
    root = tree.getroot()

    for placemark in root.iter(f'{{{ns}}}Placemark'):
        trip_id = None
        for sd in placemark.iter(f'{{{ns}}}SimpleData'):
            if sd.get('name') == 'trip_id':
                trip_id = sd.text
                break
        if trip_id is None:
            continue

        for linestring in placemark.iter(f'{{{ns}}}LineString'):
            coords_elem = linestring.find(f'{{{ns}}}coordinates')
            if coords_elem is None or not coords_elem.text:
                continue
            points = [p.strip() for p in coords_elem.text.strip().split() if p.strip()]
            all_seconds[(vague, trip_id)] += len(points)

    print(f"  → {vague} OK")


def classify_trip(trip_id):
    tid = trip_id.lower().strip()
    # Cas spéciaux
    if tid.startswith('372t'): return 'v'
    if tid.startswith('356t'): return 'v'
    if tid.startswith('359t'): return 'v'
    if tid.startswith('385t'): return 't'
    if tid.startswith('354t'): return 't'
    match = re.search(r'(\d+)([tv])', tid)
    return match.group(2) if match else 'unknown'

COL = 22
header = f"{'Trip ID':<{COL}} {'Type':>6} {'Vague':>7} {'Secondes':>10} {'Heures':>10}"
sep = "-" * len(header)

print(f"\n{header}")
print(sep)

for (vague, trip_id), seconds in sorted(all_seconds.items(), key=lambda x: (x[0][1], x[0][0])):
    typ = classify_trip(trip_id)
    hours = seconds / 3600
    print(f"{trip_id:<{COL}} {typ:>6} {vague:>7} {seconds:>10} {hours:>10.2f}h")

# ── Totaux globaux ─────────────────────────────────────────────────────────────
total_all = sum(all_seconds.values())
total_t   = sum(s for (v, tid), s in all_seconds.items() if classify_trip(tid) == 't')
total_v   = sum(s for (v, tid), s in all_seconds.items() if classify_trip(tid) == 'v')

riders_t = {(v, tid) for (v, tid), s in all_seconds.items() if classify_trip(tid) == 't' and s > 0}
riders_v = {(v, tid) for (v, tid), s in all_seconds.items() if classify_trip(tid) == 'v' and s > 0}

print("\n" + "=" * len(header))
print(f"{'TOTAL GÉNÉRAL':<{COL}} {'':>6} {'':>7} {total_all:>10} {total_all/3600:>10.2f}h")
print(f"{'TOTAL type t':<{COL}} {'t':>6} {'':>7} {total_t:>10} {total_t/3600:>10.2f}h  ({len(riders_t)} riders avec trajets)")
print(f"{'TOTAL type v':<{COL}} {'v':>6} {'':>7} {total_v:>10} {total_v/3600:>10.2f}h  ({len(riders_v)} riders avec trajets)")

# ── Totaux par vague ───────────────────────────────────────────────────────────
print()
for vague in ["vague1", "vague2", "vague3"]:
    secs_vague = {tid: s for (v, tid), s in all_seconds.items() if v == vague}
    total_v_t  = sum(s for tid, s in secs_vague.items() if classify_trip(tid) == 't')
    total_v_v  = sum(s for tid, s in secs_vague.items() if classify_trip(tid) == 'v')
    nb_t = sum(1 for tid, s in secs_vague.items() if classify_trip(tid) == 't' and s > 0)
    nb_v = sum(1 for tid, s in secs_vague.items() if classify_trip(tid) == 'v' and s > 0)
    total_vague = sum(secs_vague.values())
    print(f"{vague.upper()} — total: {total_vague/3600:.2f}h  |  "
          f"t: {total_v_t/3600:.2f}h ({nb_t} riders)  |  "
          f"v: {total_v_v/3600:.2f}h ({nb_v} riders)")

Traitement vague1...
  → vague1 OK
Traitement vague2...
  → vague2 OK
Traitement vague3...
  → vague3 OK

Trip ID                  Type   Vague   Secondes     Heures
-----------------------------------------------------------
311t                        t  vague1      10251       2.85h
314t                        t  vague1       4702       1.31h
317v                        v  vague1       4329       1.20h
323v                        v  vague1      45469      12.63h
323v                        v  vague3      65957      18.32h
324t                        t  vague1      75818      21.06h
324t                        t  vague2      39957      11.10h
332t                        t  vague1      20264       5.63h
332t                        t  vague2      46736      12.98h
332v                        v  vague3      25250       7.01h
335t                        t  vague1      24775       6.88h
335t                        t  vague2      28211       7.84h
335v                        v  vague3     

In [3]:
import xml.etree.ElementTree as ET
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point
from collections import defaultdict
import re
import math

# Chemins
kml_paths = {
    "vague1": "/Volumes/My Passport/NEWMOB/all_trips_vague1.kml",
    "vague2": "/Volumes/My Passport/NEWMOB/all_trips_vague2.kml",
    "vague3": "/Volumes/My Passport/NEWMOB/all_trips_vague3.kml",
}
geojson_paths = [
    "/Users/martin.dejaeghere/PhD/NEWMOB-main/GIS/places_paris.geojson",
    "/Users/martin.dejaeghere/PhD/NEWMOB-main/GIS/places_lyon.geojson",
    "/Users/martin.dejaeghere/PhD/NEWMOB-main/GIS/places_marseille.geojson",
]

MIN_SEGMENT_SEC = 10
MAX_GAP_SEC     = 20
MIN_SPEED_KMH   = 2.0

pedestrian_paths = gpd.GeoDataFrame(
    pd.concat([gpd.read_file(p) for p in geojson_paths], ignore_index=True),
    crs='EPSG:4326'
).to_crs(epsg=2154)
ns = 'http://www.opengis.net/kml/2.2'

def haversine_m(lon1, lat1, lon2, lat2):
    R = 6371000
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlam = math.radians(lon2 - lon1)
    a = math.sin(dphi/2)**2 + math.cos(phi1)*math.cos(phi2)*math.sin(dlam/2)**2
    return R * 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))

def classify_trip(trip_id):
    tid = trip_id.lower().strip()
    # Cas spéciaux
    if tid.startswith('372t'): return 'v'
    if tid.startswith('356t'): return 'v'
    if tid.startswith('359t'): return 'v'
    if tid.startswith('385t'): return 't'
    if tid.startswith('354t'): return 't'
    match = re.search(r'(\d+)([tv])', tid)
    return match.group(2) if match else 'unknown'

# clé : (vague, trip_id) → secondes
all_seconds = defaultdict(int)

for vague, kml_path in kml_paths.items():
    print(f"Traitement {vague}...")
    tree = ET.parse(kml_path)
    root = tree.getroot()

    for placemark in root.iter(f'{{{ns}}}Placemark'):
        trip_id = None
        for sd in placemark.iter(f'{{{ns}}}SimpleData'):
            if sd.get('name') == 'trip_id':
                trip_id = sd.text
                break
        if trip_id is None:
            continue

        for linestring in placemark.iter(f'{{{ns}}}LineString'):
            coords_elem = linestring.find(f'{{{ns}}}coordinates')
            if coords_elem is None or not coords_elem.text:
                continue

            raw_points = []
            for p in coords_elem.text.strip().split():
                parts = p.split(',')
                if len(parts) >= 2:
                    try:
                        raw_points.append((float(parts[0]), float(parts[1])))
                    except ValueError:
                        continue

            if len(raw_points) < 2:
                continue

            speeds_kmh = [0.0]
            for i in range(1, len(raw_points)):
                dist_m = haversine_m(*raw_points[i-1], *raw_points[i])
                speeds_kmh.append(dist_m * 3.6)

            points_2154 = gpd.GeoDataFrame(
                geometry=[Point(lon, lat) for lon, lat in raw_points],
                crs="EPSG:4326"
            ).to_crs(epsg=2154)

            joined = gpd.sjoin(points_2154, pedestrian_paths, how='left', predicate='intersects')
            joined = joined[~joined.index.duplicated(keep='first')]
            in_buffer = ~joined['index_right'].isna().values

            in_buffer = [
                in_buf and speeds_kmh[i] > MIN_SPEED_KMH
                for i, in_buf in enumerate(in_buffer)
            ]

            sequences = []
            current_seq = []
            gap_count = 0

            for i, inside in enumerate(in_buffer):
                if inside:
                    current_seq.append(i)
                    gap_count = 0
                else:
                    gap_count += 1
                    if gap_count > MAX_GAP_SEC:
                        if current_seq:
                            sequences.append(current_seq)
                        current_seq = []
                        gap_count = 0

            if current_seq:
                sequences.append(current_seq)

            for seq in sequences:
                if len(seq) >= MIN_SEGMENT_SEC:
                    all_seconds[(vague, trip_id)] += len(seq)

    print(f"  → {vague} OK")


# ── Affichage ──────────────────────────────────────────────────────────────────

COL = 22

header = f"{'Trip ID':<{COL}} {'Type':>6} {'Vague':>7} {'Secondes':>10} {'Heures':>10}"
sep    = "-" * len(header)

print(f"\n{header}")
print(sep)

# Trier par (trip_id, vague)
for (vague, trip_id), seconds in sorted(all_seconds.items(), key=lambda x: (x[0][1], x[0][0])):
    typ   = classify_trip(trip_id)
    hours = seconds / 3600
    print(f"{trip_id:<{COL}} {typ:>6} {vague:>7} {seconds:>10} {hours:>10.2f}h")

# ── Totaux globaux ─────────────────────────────────────────────────────────────
total_all = sum(all_seconds.values())
total_t   = sum(s for (v, tid), s in all_seconds.items() if classify_trip(tid) == 't')
total_v   = sum(s for (v, tid), s in all_seconds.items() if classify_trip(tid) == 'v')

# Nombre de riders uniques (vague, trip_id) avec au moins 1 seconde
riders_t = {(v, tid) for (v, tid), s in all_seconds.items() if classify_trip(tid) == 't' and s > 0}
riders_v = {(v, tid) for (v, tid), s in all_seconds.items() if classify_trip(tid) == 'v' and s > 0}

print("\n" + "=" * len(header))
print(f"{'TOTAL GÉNÉRAL':<{COL}} {'':>6} {'':>7} {total_all:>10} {total_all/3600:>10.2f}h")
print(f"{'TOTAL type t':<{COL}} {'t':>6} {'':>7} {total_t:>10} {total_t/3600:>10.2f}h  ({len(riders_t)} riders avec trajets)")
print(f"{'TOTAL type v':<{COL}} {'v':>6} {'':>7} {total_v:>10} {total_v/3600:>10.2f}h  ({len(riders_v)} riders avec trajets)")

# ── Totaux par vague ───────────────────────────────────────────────────────────
print()
for vague in ["vague1", "vague2", "vague3"]:
    secs_vague = {tid: s for (v, tid), s in all_seconds.items() if v == vague}
    total_v_t  = sum(s for tid, s in secs_vague.items() if classify_trip(tid) == 't')
    total_v_v  = sum(s for tid, s in secs_vague.items() if classify_trip(tid) == 'v')
    nb_t = sum(1 for tid, s in secs_vague.items() if classify_trip(tid) == 't' and s > 0)
    nb_v = sum(1 for tid, s in secs_vague.items() if classify_trip(tid) == 'v' and s > 0)
    total_vague = sum(secs_vague.values())
    print(f"{vague.upper()} — total: {total_vague/3600:.2f}h  |  "
          f"t: {total_v_t/3600:.2f}h ({nb_t} riders)  |  "
          f"v: {total_v_v/3600:.2f}h ({nb_v} riders)")


/var/folders/y0/0nrj3m412p978185q3d2503sr02q24/T/ipykernel_87977/3710327590.py:26: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  pd.concat([gpd.read_file(p) for p in geojson_paths], ignore_index=True),


Traitement vague1...
  → vague1 OK
Traitement vague2...
  → vague2 OK
Traitement vague3...
  → vague3 OK

Trip ID                  Type   Vague   Secondes     Heures
-----------------------------------------------------------
317v                        v  vague1         77       0.02h
323v                        v  vague3        232       0.06h
324t                        t  vague1       1684       0.47h
332t                        t  vague1         15       0.00h
332t                        t  vague2        155       0.04h
339v                        v  vague2         40       0.01h
355t                        t  vague1        618       0.17h
359t                        v  vague1         31       0.01h
363t                        t  vague2        154       0.04h
365t                        t  vague1        113       0.03h
367v                        v  vague1         47       0.01h
367v                        v  vague2         71       0.02h
368t                        t  vague1     

In [ ]:
import os
import cv2
import pandas as pd
import math
import re

# Coordonnées des 3 villes
CITIES = {
    "marseille": (43.2965, 5.3698),
    "lyon":      (45.7640, 4.8357),
    "paris":     (48.8566, 2.3522),
}

VAGUE_MAP = {
    "marseille": "vague1",
    "lyon":      "vague2",
    "paris":     "vague3",
}

def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlam = math.radians(lon2 - lon1)
    a = math.sin(dphi/2)**2 + math.cos(phi1)*math.cos(phi2)*math.sin(dlam/2)**2
    return R * 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))

def closest_city(lat, lon):
    return min(CITIES, key=lambda c: haversine_km(lat, lon, *CITIES[c]))

def classify_rider(rider_id):
    rid = rider_id.lower().strip()
    if rid.startswith('372t'): return 'v'
    if rid.startswith('356t'): return 'v'
    if rid.startswith('359t'): return 'v'
    if rid.startswith('385t'): return 't'
    if rid.startswith('354t'): return 't'
    match = re.search(r'(\d+)([tv])', rid)
    return match.group(2) if match else 'unknown'

root_folder = "/Volumes/My Passport/NEWMOB/WHOLE"

results = []

for root, dirs, files in os.walk(root_folder):

    # Construire un index CSV du dossier courant : basename_sans_ext → chemin
    csv_index = {}
    for f in files:
        if f.endswith(".csv") and not f.startswith("._"):
            base = os.path.splitext(f)[0]  # ex: "314t_2022-11-27 15_17_43_314t"
            csv_index[base] = os.path.join(root, f)

    for file in files:

        # Ignorer les fichiers fantômes macOS
        if file.startswith("._"):
            continue

        if not file.endswith(".mp4"):
            continue
        if file.endswith(("roll.mp4", "canny.mp4", "canny 2.mp4", "lanes.mp4")):
            continue

        video_path = os.path.join(root, file)
        cap = cv2.VideoCapture(video_path)
        fps    = cap.get(cv2.CAP_PROP_FPS)
        frames = cap.get(cv2.CAP_PROP_FRAME_COUNT)
        duration = frames / fps if fps > 0 else 0
        cap.release()

        rider_id = file.split("_")[0]

        # Chercher le CSV correspondant : même nom que le MP4 sans extension
        # ex: VID_314t_2022-11-27 15_17_43_314t.mp4 → 314t_2022-11-27 15_17_43_314t.csv
        base_mp4 = os.path.splitext(file)[0]  # sans .mp4
        # Retirer préfixe "VID_" si présent
        base_csv = base_mp4[4:] if base_mp4.startswith("VID_") else base_mp4

        lat, lon, vague = None, None, "unknown"

        if base_csv in csv_index:
            try:
                df_gps = pd.read_csv(csv_index[base_csv], sep=',', on_bad_lines='skip')
                df_gps.columns = df_gps.columns.str.strip()

                if 'Lat' in df_gps.columns and 'Long' in df_gps.columns:
                    #print('colonnes gps trouvées')
                    df_gps['Lat']  = pd.to_numeric(df_gps['Lat'],  errors='coerce')
                    df_gps['Long'] = pd.to_numeric(df_gps['Long'], errors='coerce')
                    df_gps = df_gps.dropna(subset=['Lat', 'Long'])

                    if not df_gps.empty:
                        lat   = df_gps['Lat'].mean()
                        lon   = df_gps['Long'].mean()
                        city  = closest_city(lat, lon)
                        vague = VAGUE_MAP[city]
            except Exception as e:
                print(f"  ⚠ Erreur CSV {base_csv} : {e}")
        else:
            print(f"  ⚠ CSV non trouvé pour : {file}  (cherché : {base_csv}.csv)")

        results.append({
            "rider_id":   rider_id,
            "file":       file,
            "duration_s": duration,
            "vague":      vague,
            "type":       classify_rider(rider_id),
        })

df = pd.DataFrame(results)

# ── Agrégation ─────────────────────────────────────────────────────────────────
df_grouped = df.groupby(['rider_id', 'type', 'vague']).agg(
    duration_s=('duration_s', 'sum'),
    n_rides   =('duration_s', 'count')
).reset_index()

df_grouped['duration_h'] = (df_grouped['duration_s'] / 3600).round(2)

print("\nDurée par rider :")
print(df_grouped.to_string(index=False))

# ── Statistiques globales ──────────────────────────────────────────────────────
total_riders  = df_grouped.shape[0]
total_rides   = df_grouped['n_rides'].sum()
total_seconds = df_grouped['duration_s'].sum()

print("\n===== DATASET SUMMARY =====")
print(f"Total riders  : {total_riders}")
print(f"Total rides   : {total_rides}")
print(f"Total duration: {round(total_seconds,2)}s / {round(total_seconds/3600,2)}h")

for typ in ['t', 'v']:
    sub  = df_grouped[df_grouped['type'] == typ]
    secs = sub['duration_s'].sum()
    nb   = sub['rider_id'].nunique()
    print(f"Type {typ}        : {round(secs,2)}s / {round(secs/3600,2)}h  ({nb} riders)")

for vague in ['vague1', 'vague2', 'vague3', 'unknown']:
    sub  = df_grouped[df_grouped['vague'] == vague]
    if sub.empty:
        continue
    secs = sub['duration_s'].sum()
    nb_t = sub[sub['type'] == 't']['rider_id'].nunique()
    nb_v = sub[sub['type'] == 'v']['rider_id'].nunique()
    print(f"{vague.upper():<10}: {round(secs/3600,2)}h  |  t: {nb_t} riders  |  v: {nb_v} riders")

    # ── Sauvegarde CSV ─────────────────────────────────────────────────────────────
output_folder = os.path.join(root_folder, "dataset_summary")
os.makedirs(output_folder, exist_ok=True)

csv_riders = os.path.join( "rider_summary.csv")
csv_rides  = os.path.join( "rides_detail.csv")

df_grouped.to_csv(csv_riders, index=False)
df.to_csv(csv_rides, index=False)

print(f"\nCSV riders saved to: {csv_riders}")
print(f"CSV rides  saved to: {csv_rides}")

In [2]:
import os
import cv2
import pandas as pd
import math
import re
import subprocess

root_folder = "/Volumes/My Passport/escooter"
codebook_folder = "/Volumes/My Passport/codebookescooter"

CITIES = {
    "marseille": [
        (43.2965, 5.3698),
        (43.6402, 5.0977),
        (43.5297, 5.4474),
    ],
    "lyon": [
        (45.7640, 4.8357),
    ],
    "paris": [
        (48.8566, 2.3522),
    ],
}

VAGUE_MAP = {
    "marseille": "vague1",
    "lyon":      "vague2",
    "paris":     "vague3",
}

EXCLUDED_SUFFIXES = ("roll.mp4", "canny.mp4", "canny 2.mp4", "lanes.mp4")


def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlam = math.radians(lon2 - lon1)
    a = math.sin(dphi / 2) ** 2 + math.cos(phi1) * math.cos(phi2) * math.sin(dlam / 2) ** 2
    return R * 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))


def closest_city(lat, lon):
    best_city, best_dist = None, float("inf")
    for city, coords_list in CITIES.items():
        for (clat, clon) in coords_list:
            dist = haversine_km(lat, lon, clat, clon)
            if dist < best_dist:
                best_dist = dist
                best_city = city
    return best_city


def classify_rider(rider_id):
    rid = rider_id.lower().strip()
    if rid.startswith("372t"): return "v"
    if rid.startswith("356t"): return "v"
    if rid.startswith("359t"): return "v"
    if rid.startswith("385t"): return "t"
    if rid.startswith("354t"): return "t"
    match = re.search(r"(\d+)([tv])", rid)
    return match.group(2) if match else "unknown"


def get_log_paths(video_name, codebook_folder):
    log_file_1 = os.path.join(codebook_folder, f"{video_name}_rater1_session.json")
    log_file_2 = os.path.join(codebook_folder, f"{video_name}_rater2_session.json")
    return log_file_1, log_file_2


def has_at_least_one_log(video_name, codebook_folder):
    log_file_1, log_file_2 = get_log_paths(video_name, codebook_folder)
    return os.path.isfile(log_file_1) or os.path.isfile(log_file_2), log_file_1, log_file_2


def add_yellow_finder_tag(file_path):
    """Applique le tag jaune Finder via osascript — aucune dépendance externe."""
    script = f'''
    tell application "Finder"
        set theFile to POSIX file "{file_path}" as alias
        set label index of theFile to 3
    end tell
    '''
    result = subprocess.run(["osascript", "-e", script], capture_output=True, text=True)
    if result.returncode != 0:
        print(f"  ⚠ Erreur tag sur {file_path} : {result.stderr.strip()}")


results = []

for root, dirs, files in os.walk(root_folder):

    csv_index = {}
    for f in files:
        if f.endswith(".csv") and not f.startswith("._"):
            base = os.path.splitext(f)[0]
            csv_index[base] = os.path.join(root, f)

    for file in files:
        if file.startswith("._"):
            continue
        if not file.endswith(".mp4"):
            continue
        if file.endswith(EXCLUDED_SUFFIXES):
            continue

        video_name = os.path.splitext(file)[0]

        ok_logs, log_file_1, log_file_2 = has_at_least_one_log(video_name, codebook_folder)
        if not ok_logs:
            continue

        video_path = os.path.join(root, file)

        # ── Tag Finder jaune (label index 3) si log présent ──────────────────
        print(f"  → Tag jaune : {file}")
        add_yellow_finder_tag(video_path)
        # ─────────────────────────────────────────────────────────────────────

        cap = cv2.VideoCapture(video_path)
        fps = cap.get(cv2.CAP_PROP_FPS)
        frames = cap.get(cv2.CAP_PROP_FRAME_COUNT)
        duration = frames / fps if fps and fps > 0 else 0
        cap.release()

        rider_id = file.split("_")[0]
        vehicle = classify_rider(rider_id)

        base_mp4 = os.path.splitext(file)[0]
        base_csv = base_mp4[4:] if base_mp4.startswith("VID_") else base_mp4

        vague = "unknown"

        if base_csv in csv_index:
            try:
                df_gps = pd.read_csv(csv_index[base_csv], sep=",", on_bad_lines="skip")
                df_gps.columns = df_gps.columns.str.strip()

                if "Lat" in df_gps.columns and "Long" in df_gps.columns:
                    df_gps["Lat"] = pd.to_numeric(df_gps["Lat"], errors="coerce")
                    df_gps["Long"] = pd.to_numeric(df_gps["Long"], errors="coerce")
                    df_gps = df_gps.dropna(subset=["Lat", "Long"])

                    if not df_gps.empty:
                        city = closest_city(df_gps["Lat"].mean(), df_gps["Long"].mean())
                        vague = VAGUE_MAP.get(city, "unknown")
                else:
                    print(f"  ⚠ Colonnes Lat/Long absentes dans : {csv_index[base_csv]}")

            except Exception as e:
                print(f"  ⚠ Erreur CSV {base_csv} : {e}")
        else:
            print(f"  ⚠ CSV non trouvé pour : {file}")

        results.append({
            "rider_id": rider_id,
            "vehicle": vehicle,
            "vague": vague,
            "duration_s": duration,
            "ride": 1
        })


df = pd.DataFrame(results)

print(f"\nNombre de résultats collectés : {len(results)}")

if df.empty:
    print("Aucune donnée collectée. Vérifie les logs, les vidéos et les CSV.")
else:
    print(f"Colonnes du df : {df.columns.tolist()}")
    print(df.head(10))


mp4_count = 0
log_found = 0

for root, dirs, files in os.walk(root_folder):
    for file in files:
        if file.startswith("._"):
            continue
        if not file.endswith(".mp4"):
            continue
        if file.endswith(EXCLUDED_SUFFIXES):
            continue

        mp4_count += 1
        video_name = os.path.splitext(file)[0]
        ok_logs, log_file_1, log_file_2 = has_at_least_one_log(video_name, codebook_folder)

        if ok_logs:
            log_found += 1
        else:
            print(f"  ⚠ Aucun log trouvé pour : {file}")
            print(f"    rater1 : {log_file_1}")
            print(f"    rater2 : {log_file_2}")

print(f"\nMP4 trouvés : {mp4_count}")
print(f"MP4 avec au moins 1 log : {log_found}")


if not df.empty:
    df_grouped = df.groupby(["rider_id", "vehicle", "vague"]).agg(
        duration_s=("duration_s", "sum"),
        n_rides=("ride", "sum")
    ).reset_index()

    df_grouped["duration_min"] = (df_grouped["duration_s"] / 60).round(2)
    df_grouped["duration_h"] = (df_grouped["duration_s"] / 3600).round(2)

    print("\n===== DURÉE PAR RIDER =====")
    print(df_grouped.to_string(index=False))

    total_riders = df_grouped["rider_id"].nunique()
    total_rides = df_grouped["n_rides"].sum()
    total_seconds = df_grouped["duration_s"].sum()

    print("\n===== DATASET SUMMARY =====")
    print(f"Total rides    : {total_rides}")
    print(f"Total duration : {round(total_seconds, 2)}s / {round(total_seconds / 60, 2)}min / {round(total_seconds / 3600, 2)}h")

    print("\n===== SUMMARY BY VEHICLE =====")
    df_vehicle = df.groupby("vehicle").agg(
        duration_s=("duration_s", "sum"),
        n_rides=("ride", "sum"),
        n_riders=("rider_id", "nunique")
    ).reset_index()

    df_vehicle["duration_min"] = (df_vehicle["duration_s"] / 60).round(2)
    df_vehicle["duration_h"] = (df_vehicle["duration_s"] / 3600).round(2)
    print(df_vehicle.to_string(index=False))

    print("\n===== SUMMARY BY VAGUE =====")
    for vague in ["vague1", "vague2", "vague3", "unknown"]:
        sub = df_grouped[df_grouped["vague"] == vague]
        if sub.empty:
            continue

        secs = sub["duration_s"].sum()
        nb_t = sub[sub["vehicle"] == "t"]["rider_id"].nunique()
        nb_v = sub[sub["vehicle"] == "v"]["rider_id"].nunique()

        print(f"{vague.upper():<10}: {round(secs / 3600, 2)}h  |  t: {nb_t} riders  |  v: {nb_v} riders")

  → Tag jaune : 392t_2023-05-05 15_09_07_392t_0_52_3_42.mp4
  → Tag jaune : 332t_2023-05-01 15_32_57_332t_0_06_0_50.mp4
  → Tag jaune : 332t_2023-05-27 16_56_51_332t_6_21_7_11.mp4
  → Tag jaune : 335t_2023-06-02 08_06_18_335t_17_34_21_08.mp4
  → Tag jaune : 355t_2022-10-13 13_53_13_355t_0_04_0_40.mp4
  → Tag jaune : 363t_2023-06-07 13_49_04_363t_1_02_3_25.mp4
  → Tag jaune : 364t_2023-05-11 17_07_21_364t_6_11_7_20.mp4
  → Tag jaune : 369t_2023-06-01 07_52_16_369t_5_58_6_37.mp4
  → Tag jaune : 371t_2023-10-19_18_05_20_371t_0_09_1_34.mp4
  → Tag jaune : 392t_2022-10-06 11_05_52_392t_5_49_6_16.mp4
  → Tag jaune : 332t_2023-04-26 12_05_55_332t_21_27_22_15.mp4
  → Tag jaune : 332t_2023-05-27 13_23_17_332t_3_13_3_58.mp4
  → Tag jaune : 391t_2023-05-16 17_35_20_391t_17_21_18_13.mp4
  → Tag jaune : 391t_2022-10-09 16_42_09_391t_1_57_4_03.mp4
  → Tag jaune : 335t_2023-04-28 08_04_24_335t_19_56_23_32.mp4
  → Tag jaune : 335t_2023-06-02 08_06_18_335t_16_23_17_04.mp4
  → Tag jaune : 389t_2023-05-2